In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

In [3]:
# Load final processed data and existing models
df = pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\final_processed_data.csv')
print(f" Data Loaded: {df.shape}")
print(df.columns)


 Data Loaded: (33029, 30)
Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role'],
      dtype='object')


In [9]:

df['isBoundary'] = df['isFour'].astype(int) | df['isSix'].astype(int)


adaptability = df.groupby(['Full Name', 'Bowler_Bowling_Style']).agg({
    'run': 'sum',
    'isBoundary': 'mean',
    'isWicket': 'mean'
}).reset_index()


batsman_group = adaptability.groupby('Full Name').agg({
    'run': ['mean', 'std'],
    'isBoundary': ['mean', 'std'],
    'isWicket': ['mean', 'std']
})
batsman_group.columns = ['_'.join(col) for col in batsman_group.columns]
batsman_group = batsman_group.reset_index()


batsman_group['adaptability_index'] = 1 / (1 + batsman_group['run_std'].fillna(0))
batsman_group['adaptability_level'] = pd.qcut(
    batsman_group['adaptability_index'], q=3, labels=['Low', 'Medium', 'High']
)

print(" Adaptability model built successfully!")
print(batsman_group.head())


 Adaptability model built successfully!
        Full Name   run_mean    run_std  isBoundary_mean  isBoundary_std  \
0  AB de Villiers  14.625000  12.176529         0.207292        0.216844   
1    Aamir Kaleem   0.000000   0.000000         0.000000        0.000000   
2     Aaron Finch  27.750000  23.798873         0.097401        0.075421   
3   Aaron Johnson  16.833333   9.453394         0.231376        0.176339   
4     Aaron Jones  15.200000  12.951190         0.202265        0.162685   

   isWicket_mean  isWicket_std  adaptability_index adaptability_level  
0       0.047569      0.076166            0.075893                Low  
1       0.500000      0.500000            1.000000               High  
2       0.032811      0.036030            0.040324                Low  
3       0.033704      0.037723            0.095663                Low  
4       0.020000      0.063246            0.071678                Low  


In [10]:

def get_phase(over):
    if over <= 6: return 'Powerplay'
    elif over <= 15: return 'Middle'
    else: return 'Death'

df['match_phase'] = df['oversActual'].apply(get_phase)

phase_sr = df.groupby(['Full Name', 'match_phase']).agg({
    'run': 'sum', 'inningNumber': 'count'
}).reset_index()
phase_sr['strike_rate'] = (phase_sr['run'] / phase_sr['inningNumber']) * 100


In [11]:
def suggest_field_zone(row):
    """
    Suggests realistic field placements based on
    pitch line, length, bowler type, batsman style, and match phase.
    """
    line = row.get('pitchLine', '')
    length = row.get('pitchLength', '')
    bowler = row.get('Bowler_Bowling_Style', '').lower()
    batsman = row.get('Batsman_Batting_Style', '').lower()
    phase = 'Powerplay' if row['oversActual'] <= 6 else 'Middle' if row['oversActual'] <= 15 else 'Death'
    

    fielders = []

    
    if phase == 'Powerplay':
        fielders += ["mid-off", "mid-on", "cover", "square leg", "fine leg"]
    elif phase == 'Middle':
        fielders += ["deep mid-wicket", "long-off", "long-on", "cover", "point"]
    else:  # Death
        fielders += ["deep square leg", "third man", "long-off", "long-on", "deep extra cover"]

    
    if line in ['OUTSIDE_OFF', 'ON_THE_STUMPS']:
        if length in ['GOOD_LENGTH', 'SHORT_OF_A_GOOD_LENGTH']:
            fielders += ["cover point", "deep extra cover", "third man"]
        elif length in ['FULL', 'YORKER']:
            fielders += ["long-off", "long-on", "mid-wicket"]
        elif length in ['SHORT', 'BOUNCER']:
            fielders += ["deep square leg", "deep fine leg"]
    elif line in ['LEG_STUMP', 'OUTSIDE_LEG']:
        if length in ['GOOD_LENGTH', 'SHORT_OF_A_GOOD_LENGTH']:
            fielders += ["deep square leg", "deep mid-wicket"]
        elif length in ['FULL', 'YORKER']:
            fielders += ["long-on", "deep mid-wicket"]
        else:
            fielders += ["fine leg", "square leg"]

   
    if "offbreak" in bowler or "slow" in bowler or "spinner" in bowler:
        fielders += ["short mid-wicket", "slip"]
    elif "fast" in bowler or "medium" in bowler:
        fielders += ["third man", "fine leg"]

    
    if 'left' in batsman:
        # Swap leg/off side
        fielders = [pos.replace("off", "temp").replace("leg", "off").replace("temp", "leg") for pos in fielders]

    
    
    if row.get('isBoundary', 0) == 1:
        fielders += ["move deep fielders wider (boundary prevention)"]
    elif row.get('isWicket', 0) == 1:
        fielders += ["attack with slip, short mid-wicket"]

    
    # Remove duplicates & join
    unique_fielders = list(dict.fromkeys(fielders))  # preserve order
    return ", ".join(unique_fielders) if unique_fielders else "Balanced field"
    

# Apply enhanced logic
df['suggested_field'] = df.apply(suggest_field_zone, axis=1)
print(" Enhanced field logic applied successfully!")


 Enhanced field logic applied successfully!


In [12]:
def get_recommendation(batsman_name, phase, bowler_type="right-arm fast"):
    """
    Enhanced bowling recommendation system using adaptability, strike rate,
    phase context, and bowler type.
    """

   
    adapt_row = batsman_group[batsman_group['Full Name'].str.contains(batsman_name, case=False, na=False)]
    phase_row = phase_sr[
        (phase_sr['Full Name'].str.contains(batsman_name, case=False, na=False)) &
        (phase_sr['match_phase'] == phase)
    ]

    if adapt_row.empty or phase_row.empty:
        return f" No sufficient data for {batsman_name} in {phase} phase."

    adaptability = float(adapt_row['adaptability_index'])
    strike_rate = float(phase_row['strike_rate'])

    
    # Aggressiveness metric
    aggressiveness = (strike_rate / 150) * (1 + (1 - adaptability))
    # Determine difficulty tier
    difficulty = "High Threat" if aggressiveness > 1.2 else "Medium Threat" if aggressiveness > 0.9 else "Low Threat"

    
    if difficulty == "High Threat":
        if "fast" in bowler_type:
            line = "OUTSIDE_OFF"
            length = "SHORT_OF_A_GOOD_LENGTH"
            plan = "Keep it back of a length, attack with short balls and tight off-stump line."
        else:
            line = "ON_THE_STUMPS"
            length = "FULL"
            plan = "Use flight and drift, tempt drive — cover + long-off deep."
    elif difficulty == "Medium Threat":
        if adaptability > 0.6:
            line = "ON_THE_STUMPS"
            length = "GOOD_LENGTH"
            plan = "Stick to classic good-length channels; mix in slower balls."
        else:
            line = "OUTSIDE_OFF"
            length = "FULL"
            plan = "Pitch it fuller, outside off; use swing and variation."
    else:  # Low threat
        line = "ON_THE_STUMPS"
        length = "YORKER"
        plan = "Attack with full, straight deliveries; slip in yorkers to finish."

    
    if phase == "Powerplay":
        plan += " Restrict boundaries; field up close with attacking slips."
    elif phase == "Middle":
        plan += " Maintain pressure; rotate bowlers and change pace subtly."
    else:  # Death overs
        plan += " Focus on yorkers and wide lines; deep square leg, third man back."

    
    if "fast" in bowler_type:
        field_plan = "Deep square leg, third man, cover point, long-off, long-on."
    elif "offbreak" in bowler_type:
        field_plan = "Slip, short mid-wicket, long-on, long-off, deep mid-wicket."
    else:
        field_plan = "Balanced: cover, mid-off, deep mid-wicket, square leg."

    
    recommendation = {
        " Batsman": batsman_name,
        " Phase": phase,
        " Bowler Type": bowler_type,
        " Adaptability Index": round(adaptability, 3),
        " Strike Rate": round(strike_rate, 1),
        " Threat Level": difficulty,
        " Recommended Line": line,
        " Recommended Length": length,
        " Tactical Plan": plan,
        " Field Setup": field_plan
    }

    return recommendation


In [13]:
get_recommendation("Virat Kohli", "Middle", "right-arm fast")


C:\Users\yaswa\AppData\Local\Temp\ipykernel_9480\691522191.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  adaptability = float(adapt_row['adaptability_index'])
C:\Users\yaswa\AppData\Local\Temp\ipykernel_9480\691522191.py:18: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  strike_rate = float(phase_row['strike_rate'])


{' Batsman': 'Virat Kohli',
 ' Phase': 'Middle',
 ' Bowler Type': 'right-arm fast',
 ' Adaptability Index': 0.026,
 ' Strike Rate': 116.7,
 ' Threat Level': 'High Threat',
 ' Recommended Line': 'OUTSIDE_OFF',
 ' Recommended Length': 'SHORT_OF_A_GOOD_LENGTH',
 ' Tactical Plan': 'Keep it back of a length, attack with short balls and tight off-stump line. Maintain pressure; rotate bowlers and change pace subtly.',
 ' Field Setup': 'Deep square leg, third man, cover point, long-off, long-on.'}